#Base de Datos (Database)

In [0]:
%run "../includes/configuration"

In [0]:
%sql
--1. Documentacion Spark SQL
--https://spark.apache.org/docs/latest/sql-ref-syntax-ddl-create-database.html


In [0]:
%sql
--2. Crear la Base de datos "demo"

CREATE SCHEMA IF NOT EXISTS demo
COMMENT "Base de datos de prueba";


In [0]:
%sql
--3. Acceder al "Catalog" en la interfaz de usuario
--Esta base de datos se creo en js_ws_databricks_us, debido a que se coloco "single user" al momento de crear el cluster

In [0]:
%sql
--4. Comando "SHOW"
show databases;

In [0]:
%sql
--5. Comando DESCRIBE(DESC)
describe database demo;


In [0]:
%sql
describe database extended demo;

In [0]:
%sql
--6. Monstrar la base de datos Actual

--en esta parte nos encontramos en la base de datos default
SELECT current_database();


In [0]:
%sql
--para entrar a la base de datos que creamos

USE demo;

SELECT current_database();

In [0]:
%sql
--podemos consultar las tablas de otra base de datos a pesar de estar en demo
show tables in default;

#Tablas Administradas (Managed Tables)

In [0]:
#1. Crear tabla administrada (managed tables) con python

results_movie_genre_language_df = spark.read.parquet(f"{gold_folder_path}/results_movie_genre_language")

#creamos la tabla results_movie_genre_language_python en la base de datos demo
results_movie_genre_language_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("demo.results_movie_genre_language_python")


In [0]:
%sql
--consultamos la tabla
USE demo;
--SHOW TABLES;

describe extended demo.results_movie_genre_language_python;

In [0]:
%sql

CREATE TABLE demo.results_movie_genre_language_sql as
select * 
from demo.results_movie_genre_language_python 
where genre_name = "Adventure"

In [0]:
%sql
select * from results_movie_genre_language_sql

In [0]:
%sql
select current_database()

In [0]:
%sql
describe extended demo.results_movie_genre_language_sql

In [0]:
%sql
--3. Efecto de eliminar una tabla administrada
drop table if exists demo.results_movie_genre_language_sql

#Tablas externas


In [0]:
#1. crear una "tabla externa (External Table)" con python

results_movie_genre_language_df = spark.read.parquet(f"{gold_folder_path}/results_movie_genre_language")

#creamos la tabla results_movie_genre_language_python en la base de datos demo
results_movie_genre_language_df.write \
    .format("delta") \
    .option("path", f"{gold_folder_path}/results_movie_genre_language_py") \
    .mode("overwrite") \
    .saveAsTable("demo.results_movie_genre_language_py")

In [0]:
%sql
describe extended  demo.results_movie_genre_language_py

In [0]:
%sql
--2. crear una "tabla externa (External Table)" con SQL

create table demo.results_movie_genre_language_sql(
    title string,
    duration_time int,
    release_date date,
    vote_average float,
    language_name string,
    genre_name string,
    ingestion_date timestamp,
    env string
)
using delta
location "abfss://gold@lsdata01.dfs.core.windows.net/results_movie_genre_language_ext_sql"



In [0]:
%sql
insert into demo.results_movie_genre_language_sql
select title,  duration_time, release_date, vote_average, language_name, genre_name, ingestion_date, env
from demo.results_movie_genre_language_py
where genre_name = "Adventure"
    


In [0]:
%sql 
show tables in demo

In [0]:
%sql
--3. Efecto de elimnacion de uina tabla "tabla external (external TAble)"

drop table demo.results_movie_genre_language_sql

In [0]:
#4. Describir (describe) la tabla

##Vistas (Views)

In [0]:
%sql
--Crear vista temporal
use demo;

create or replace temp view v_results_movie_genre_language as
select * 
from demo.results_movie_genre_language_py a
where a.genre_name = "Adventure";



In [0]:
%sql
select * from v_results_movie_genre_language;

In [0]:
%sql
--Crear vista Temporal Global
use demo;

create or replace global temp view gv_results_movie_genre_language as
select * 
from demo.results_movie_genre_language_py a
where a.genre_name = "Drama";


In [0]:
%sql
show tables in global_temp;

In [0]:
%sql

select * from global_temp.gv_results_movie_genre_language;

In [0]:
%sql
use demo;

create or replace view pv_results_movie_genre_language as
select * 
from demo.results_movie_genre_language_py a
where a.genre_name = "Comedy";

In [0]:
%sql
show tables in demo;